# 07 — Huấn luyện ML-XC
Mô hình học hiệu chỉnh `ΔE = ∫ ρ · f_θ(ρ, s) dr` lên PBE để khớp nhãn `Δ = E_ref − E_PBE` (đơn vị eV/nguyên tử).

**Quy tắc dữ liệu:** chỉ **train** dùng để fit tham số; **validation** dùng để dừng sớm và chọn mô hình/siêu tham số; **test không được nạp ở notebook này**.

**Đầu ra:** `data/models/best_model.npz`, `data/models/train_log.json`.

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q h5py optax matplotlib

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd, time
import matplotlib.pyplot as plt
import jax
from utils import data_utils as dtu, density_utils as du, model_utils as mu

H5 = f"{DATA}/ml_xc_dataset.h5"
MODELS = f"{DATA}/models"
os.makedirs(MODELS, exist_ok=True)
print("JAX devices:", jax.devices())

# lưới siêu tham số nhỏ; model selection chỉ dựa trên RMSE validation
GRID = [{"hidden": (16, 16), "lr": 3e-3}, {"hidden": (32, 32), "lr": 1e-3}, {"hidden": (32, 32), "lr": 3e-3}]
EPOCHS, BATCH, PATIENCE, SEED = 300, 8, 40, 0

In [ ]:
def load_split(split):
    ids = dtu.list_materials(H5, split)
    smp = [dtu.load_sample(H5, m) for m in ids]
    return mu.make_dataset(smp, du.ml_features)

train_ds, val_ds = load_split("train"), load_split("val")      # không có "test" ở đây
print(f"train {len(train_ds)} mẫu | val {len(val_ds)} mẫu")
tg = np.array([d["target_eV"] for d in train_ds])
print(f"Nhãn train: mean {tg.mean()*1e3:.1f}, std {tg.std()*1e3:.1f} meV/nguyên tử")
base_val = mu.metrics(np.zeros(len(val_ds)), [d["target_eV"] for d in val_ds])
print("Baseline PBE (ΔE=0) trên validation:", {k: round(v, 2) for k, v in base_val.items() if k != "n"})

In [ ]:
results = []
for i, g in enumerate(GRID):
    print(f"\n=== cấu hình {i}: {g} ===")
    t0 = time.time()
    params, log = mu.train(train_ds, val_ds, hidden=g["hidden"], lr=g["lr"], epochs=EPOCHS,
                           batch_size=BATCH, patience=PATIENCE, seed=SEED)
    results.append({"cfg": g, "params": params, "log": log, "val_rmse_meV": log["best_val_rmse_meV"]})
    print(f"val RMSE tốt nhất: {log['best_val_rmse_meV']:.2f} meV/nguyên tử ({time.time()-t0:.0f}s)")
best = min(results, key=lambda r: r["val_rmse_meV"])
print("\nChọn:", best["cfg"], "| val RMSE =", round(best["val_rmse_meV"], 2), "meV/nguyên tử")

In [ ]:
mu.save_model(f"{MODELS}/best_model.npz", best["params"], {"hidden": list(best["cfg"]["hidden"]), "lr": best["cfg"]["lr"],
              "val_rmse_meV": best["val_rmse_meV"], "seed": SEED, "features": ["log10(rho)", "asinh(s)"]})
json.dump([{"cfg": {"hidden": list(r["cfg"]["hidden"]), "lr": r["cfg"]["lr"]}, "val_rmse_meV": r["val_rmse_meV"],
            "history": r["log"]["history"]} for r in results],
          open(f"{MODELS}/train_log.json", "w"), indent=1, default=float)

plt.figure(figsize=(6, 4))
for r in results:
    h = pd.DataFrame(r["log"]["history"])
    plt.plot(h.epoch, h.val_rmse_meV, label=f"{r['cfg']['hidden']} lr={r['cfg']['lr']}")
plt.axhline(base_val["rmse_meV"], color="k", ls="--", label="PBE (ΔE=0)")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("val RMSE (meV/nguyên tử)"); plt.legend(); plt.tight_layout()
plt.savefig(f"{MODELS}/val_curves.png", dpi=120); plt.show()